# Phase 1 Canonical Dataset and Performance Frame
This notebook documents the canonical dataset contract for the YouTube trending project and the default video-country snapshot used in later phases.

## Raw data inventory
The project starts from the 10 country CSV files in `data/*.csv`. Country is derived from the filename prefix and preserved through every downstream artifact.

In [ ]:
from pathlib import Path
import pandas as pd

raw_files = sorted(Path('data').glob('*videos.csv'))
inventory = pd.DataFrame({
    'country': [path.name[:2] for path in raw_files],
    'source_file': [str(path) for path in raw_files],
})
inventory

## Canonical row-level table
The canonical row-level table keeps one row per raw trending observation. Raw text fields stay intact while helper columns standardize dates, booleans, tag parsing, and missing-value flags.

In [ ]:
from youtube_trends.canonical_dataset import build_trending_rows

rows_path = Path('data/processed/trending_rows.parquet')
if not rows_path.exists():
    build_trending_rows()

rows = pd.read_parquet(rows_path)
rows[['country', 'video_id', 'trending_date_local', 'publish_time', 'tag_count']].head()

## Video-country snapshot table
The default analysis table keeps one row per `country + video_id` pair, anchored on the first local trending observation for that country. Multi-country videos remain separate rows instead of being collapsed into a single global timeline.

In [ ]:
from youtube_trends.performance_frame import write_video_country_snapshot

snapshot_path = Path('data/processed/video_country_snapshot.parquet')
if not snapshot_path.exists():
    write_video_country_snapshot()

snapshot = pd.read_parquet(snapshot_path)
snapshot[['country', 'video_id', 'first_trending_date_local', 'trend_days_in_country_proxy', 'publish_to_first_trend_days']].head()

## Trending-corpus performance proxy
The primary Phase 1 target is `trend_days_in_country_proxy`, defined as the number of row-level trending observations for a given `country + video_id` pair. This proxy stays inside the trending corpus and supports statements about association rather than causal claims about what made a video trend. Views, likes, dislikes, and comments remain secondary context metrics captured from the first trending snapshot.

## Limitations
This dataset only contains videos that were already trending, so the analysis can describe association inside the trending corpus but not causal effects. Human-readable category labels are unresolved in the current repository, and video duration is unavailable in the provided dataset. Those limits must be carried into later reporting.

## Re-run instructions
Regenerate the cached Phase 1 artifacts from the repo root with:

```bash
python -m youtube_trends.canonical_dataset
python -m youtube_trends.performance_frame
```

The first command rebuilds the row-level canonical parquet and source manifest. The second command rebuilds the video-country snapshot parquet from the canonical row-level table.